# Lab 5 — Ground the Assistant with Azure AI Search

**Required · 45 minutes · Level 200**

Connect an ephemeral Agent Framework agent to the prepared VWI/manual index.
This lab is intentionally read-only: shared workshop indexes are provisioned
before the session and must not be deleted by participants.

## Learning objectives

- Resolve a Foundry project connection to Azure AI Search.
- Attach the current Azure AI Search agent tool.
- Require citations and an explicit “I don't know” fallback.
- Compare a direct index tool with the managed knowledge-base approach in Lab 6.

In [ ]:
import os
import re
import time

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import HttpResponseError
from azure.identity import AzureCliCredential

SEARCH_CONNECTION_NAME = os.environ["AZURE_SEARCH_CONNECTION_NAME"]
SEARCH_INDEX_NAME = os.getenv("LAB_SEARCH_INDEX") or os.environ["AZURE_SEARCH_INDEX"]

credential = AzureCliCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)


def resolve_connection_id(connection_name: str, attempts: int = 6) -> str:
    for attempt in range(1, attempts + 1):
        try:
            matches = [
                item for item in project.connections.list()
                if item.name == connection_name
            ]
        except HttpResponseError as exc:
            transient = exc.status_code == 404 and 'project not found' in str(exc).lower()
            if not transient or attempt == attempts:
                raise
            matches = []
        if len(matches) == 1:
            return matches[0].id
        if len(matches) > 1:
            raise RuntimeError(
                f'Expected one project connection named {connection_name!r}; found {len(matches)}.'
            )
        if attempt == attempts:
            raise RuntimeError(
                f'Project connection {connection_name!r} was not visible after {attempts} attempts.'
            )
        delay = min(5 * (2 ** (attempt - 1)), 30)
        print(f'Resolve Search connection: not visible yet; retrying in {delay}s')
        time.sleep(delay)


connection_id = resolve_connection_id(SEARCH_CONNECTION_NAME)
print(f"Search connection: {SEARCH_CONNECTION_NAME}")
print(f"Read-only index: {SEARCH_INDEX_NAME}")

## Participant task

Edit `QUESTION` so that it asks for both a maintenance procedure and the
evidence an operator should record. Keep it generic and synthetic.

In [ ]:
search_tool = FoundryChatClient.get_azure_ai_search_tool(
    index_connection_id=connection_id,
    index_name=SEARCH_INDEX_NAME,
    query_type="semantic",
    top_k=5,
)
search_agent = Agent(
    client=FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL,
        credential=credential,
    ),
    name=f"search-grounded-ops-{RESOURCE_NAMESPACE}",
    instructions=(
        "Use the Azure AI Search tool for every factual answer. Cite the "
        "retrieved sources. If the index does not support the answer, say "
        "\"I don't know based on the indexed material.\" Do not invent a "
        "procedure and do not issue switching commands."
    ),
    # azure-ai-projects model objects must be converted for the in-process
    # Agent request payload with the pinned SDK versions.
    tools=[search_tool.as_dict()],
)

# TODO(participant): ask a second question that should return “I don't know”.
QUESTION = (
    "What is VWI E-85 used for, and what evidence should an operator "
    "record before escalating?"
)
result = await search_agent.run(QUESTION)
print(result.text)

## Inspect citation annotations

In [ ]:
def collect_citation_urls(agent_result) -> list[str]:
    urls: list[str] = []
    for message in agent_result.messages:
        for content in message.contents:
            for annotation in getattr(content, "annotations", None) or []:
                url = (
                    annotation.get("url")
                    if isinstance(annotation, dict)
                    else getattr(annotation, "url", None)
                )
                if url:
                    urls.append(url)
    return list(dict.fromkeys(urls))


MAX_CITATION_ATTEMPTS = 3
citation_attempt = 1
citation_urls = collect_citation_urls(result)
while not citation_urls and citation_attempt < MAX_CITATION_ATTEMPTS:
    citation_attempt += 1
    print(f"Citation annotations missing; retrying ({citation_attempt}/{MAX_CITATION_ATTEMPTS}).")
    result = await search_agent.run(QUESTION)
    print(result.text)
    citation_urls = collect_citation_urls(result)

print(f"Citation URLs found: {len(citation_urls)} after {citation_attempt} attempt(s)")
for url in citation_urls:
    print(f"- {url}")

## Deterministic success check

In [ ]:
assert result.text.strip(), "The grounded agent returned no text."
assert SEARCH_INDEX_NAME, "LAB_SEARCH_INDEX resolved to an empty name."
assert RESOURCE_NAMESPACE in search_agent.name
assert citation_urls, (
    f"No machine-readable citation URLs after {MAX_CITATION_ATTEMPTS} attempts. "
    "Verify that the Search index exposes a retrievable source URL field."
)
assert all(url.startswith("https://") for url in citation_urls), citation_urls
print("PASS — the answer contains machine-readable citation URLs.")

## Cleanup safety

There is no deletion cell. This notebook creates an in-process agent and reads
an existing index. Never add unconditional index deletion to a participant lab.

## Optional extension

Compare `simple` with `semantic` on the same three questions.

**Expected artifact:** a grounded answer plus citation inspection output.